# Assignments 2.7, 2.8, 2.9 — Flux / Qwen-Image

Notebook này chạy trên Kaggle GPU khi cần cho bài 2.7. Bật Internet và thêm Kaggle secret HF_TOKEN cho phần Diffusers của 2.7.

- 2.7: Diffusers inference cho Flux.1, Flux.2 và Qwen-Image.
- 2.8: Dựng workflow trong ComfyUI và export API-format JSON; không chạy inference.
- 2.9: Dựng workflow ComfyUI Flux Kontext với hai ảnh tham chiếu và export API-format JSON; không chạy inference.

In [ ]:
!pip -q install -U torchao==0.16.0 git+https://github.com/huggingface/diffusers.git transformers accelerate sentencepiece safetensors bitsandbytes

import gc
import os
from pathlib import Path
import torch

assert torch.cuda.is_available(), 'Trong Kaggle hãy chọn Accelerator = GPU.'
gpu_major, _ = torch.cuda.get_device_capability()
assert gpu_major >= 7, 'P100 (sm_60) không tương thích PyTorch Kaggle hiện tại. Hãy chọn T4 x2, L4 hoặc A100.'

try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
except Exception as error:
    raise RuntimeError('Không đọc được Kaggle Secret HF_TOKEN.') from error

os.environ['HF_TOKEN'] = HF_TOKEN
DTYPE = torch.float16
print(torch.cuda.get_device_name(0), DTYPE)

def free_memory(pipe):
    del pipe
    gc.collect()
    torch.cuda.empty_cache()

## Assignment 2.7 — Diffusers inference

In [ ]:
# FLUX.1 schnell: 4-bit + sequential CPU offload avoids cross-GPU 4-bit matmul.
from diffusers import BitsAndBytesConfig as DiffusersBitsAndBytesConfig
from diffusers import FluxPipeline, FluxTransformer2DModel
from transformers import BitsAndBytesConfig, T5EncoderModel

for name in ('pipe', 'transformer', 'text_encoder_2'):
    globals().pop(name, None)
gc.collect()
torch.cuda.empty_cache()

MODEL_ID = 'black-forest-labs/FLUX.1-schnell'
quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=DTYPE)
diffusers_quant = DiffusersBitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=DTYPE)

text_encoder_2 = T5EncoderModel.from_pretrained(
    MODEL_ID, subfolder='text_encoder_2', quantization_config=quant,
    torch_dtype=DTYPE, low_cpu_mem_usage=True,
)
transformer = FluxTransformer2DModel.from_pretrained(
    MODEL_ID, subfolder='transformer', quantization_config=diffusers_quant,
    torch_dtype=DTYPE, low_cpu_mem_usage=True,
)
pipe = FluxPipeline.from_pretrained(
    MODEL_ID, text_encoder_2=text_encoder_2, transformer=transformer,
    torch_dtype=DTYPE, low_cpu_mem_usage=True,
)
pipe.enable_model_cpu_offload()

image = pipe(
    prompt='A small red scooter parked in a quiet Hanoi street after rain, realistic photo',
    num_inference_steps=4, guidance_scale=0.0, max_sequence_length=256,
    height=512, width=512, generator=torch.Generator('cuda').manual_seed(42),
).images[0]
image.save('/kaggle/working/flux1.png')
display(image)
del pipe, transformer, text_encoder_2
gc.collect()
torch.cuda.empty_cache()

In [ ]:
# Flux.2 Klein 4B: text-to-image (Flux.2 bản nhỏ hơn, phù hợp Kaggle hơn Flux.2 Dev)
from diffusers import Flux2KleinPipeline

pipe = Flux2KleinPipeline.from_pretrained(
    'black-forest-labs/FLUX.2-klein-4B', torch_dtype=DTYPE
)
pipe.enable_model_cpu_offload()

image = pipe(
    prompt='A ceramic cup of Vietnamese coffee on a wooden table, soft morning light',
    num_inference_steps=4,
    guidance_scale=4.0,
    height=768, width=1024,
    generator=torch.Generator('cuda').manual_seed(42),
).images[0]
image.save('/kaggle/working/flux2.png')
display(image)
free_memory(pipe)

In [ ]:
# Qwen-Image NF4 inference on two T4 GPUs.
from diffusers import (
    QwenImagePipeline, QwenImageTransformer2DModel, AutoencoderKLQwenImage,
    FlowMatchEulerDiscreteScheduler,
)
from transformers import Qwen2_5_VLForConditionalGeneration, Qwen2Tokenizer
import bitsandbytes as bnb

MODEL_ID = 'diffusers/qwen-image-nf4'
transformer_map = {
    'pos_embed': 0, 'time_text_embed': 0, 'txt_norm': 0,
    'img_in': 0, 'txt_in': 0, 'norm_out': 1, 'proj_out': 1,
}
transformer_map.update({f'transformer_blocks.{i}': 0 if i < 30 else 1 for i in range(60)})
transformer = QwenImageTransformer2DModel.from_pretrained(
    MODEL_ID, subfolder='transformer', dtype=DTYPE,
    device_map=transformer_map, low_cpu_mem_usage=True,
)
text_map = {
    'model.visual.patch_embed': 0, 'model.visual.rotary_pos_emb': 0,
    'model.visual.merger': 1,
    'model.language_model.embed_tokens': 0,
    'model.language_model.rotary_emb': 0,
    'model.language_model.norm': 1, 'lm_head': 1,
}
text_map.update({f'model.visual.blocks.{i}': 0 if i < 16 else 1 for i in range(32)})
text_map.update({f'model.language_model.layers.{i}': 0 if i < 14 else 1 for i in range(28)})
text_encoder = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID, subfolder='text_encoder', dtype=DTYPE,
    device_map=text_map, low_cpu_mem_usage=True,
)
vae = AutoencoderKLQwenImage.from_pretrained(
    MODEL_ID, subfolder='vae', dtype=DTYPE, low_cpu_mem_usage=True
).to('cuda:0')
tokenizer = Qwen2Tokenizer.from_pretrained(MODEL_ID, subfolder='tokenizer')
scheduler = FlowMatchEulerDiscreteScheduler.from_pretrained(MODEL_ID, subfolder='scheduler')
pipe = QwenImagePipeline(
    scheduler=scheduler, vae=vae, text_encoder=text_encoder,
    tokenizer=tokenizer, transformer=transformer,
)
# The NF4 checkpoint computes in BF16, so its non-quantized layers use BF16 too.
for module in transformer.modules():
    if not any(module.children()) and not isinstance(module, bnb.nn.Linear4bit):
        module.to(dtype=torch.bfloat16)
vae.to(dtype=torch.bfloat16)
text_encoder.lm_head.to(dtype=torch.bfloat16)

image = pipe(
    prompt='A clean poster saying "KAGGLE AI" in blue letters, a robot holding a notebook',
    num_inference_steps=4, guidance_scale=0.0,
    height=256, width=256,
    generator=torch.Generator('cuda:0').manual_seed(42),
).images[0]
out_path = Path('/kaggle/working/qwen_image.png')
image.save(out_path)
print(f'QWEN_OK: saved {out_path.name} ({out_path.stat().st_size} bytes)')
display(image)
del pipe, transformer, text_encoder, vae, tokenizer, scheduler
gc.collect()
torch.cuda.empty_cache()

## Assignment 2.8 — ComfyUI workflow API JSON

ComfyUI is used here only to design the node graph and export API-format JSON. No model download, ComfyUI server, or image inference is needed for this assignment.

The three small workflows are saved as:

- /kaggle/working/flux1_api.json
- /kaggle/working/flux2_api.json
- /kaggle/working/qwen_image_api.json

Each JSON can be imported into ComfyUI or sent to its /prompt endpoint when execution is required.

In [ ]:
# ComfyUI is used only to describe/export the workflow here.
# No model download, server startup, or inference is required.
import json
from pathlib import Path

def save_workflow(workflow, name):
    out = Path(f'/kaggle/working/{name}_api.json')
    out.write_text(json.dumps(workflow, indent=2))
    print(f'WORKFLOW_JSON: {out}')
    print('Nodes:', len(workflow))

In [ ]:
# Helper only: save API-format JSON. It does not queue or run a prompt.

In [ ]:
# Flux.1 workflow: loader -> text -> sampler -> VAE decode -> save.
flux1_workflow = {
    '1': {'class_type': 'UNETLoader', 'inputs': {'unet_name': 'flux1-schnell-fp8.safetensors', 'weight_dtype': 'default'}},
    '2': {'class_type': 'DualCLIPLoader', 'inputs': {'clip_name1': 't5xxl_fp8_e4m3fn.safetensors', 'clip_name2': 'clip_l.safetensors', 'type': 'flux'}},
    '3': {'class_type': 'VAELoader', 'inputs': {'vae_name': 'ae.safetensors'}},
    '4': {'class_type': 'CLIPTextEncode', 'inputs': {'text': 'A small red scooter parked in a quiet Hanoi street after rain, realistic photo', 'clip': ['2', 0]}},
    '5': {'class_type': 'CLIPTextEncode', 'inputs': {'text': '', 'clip': ['2', 0]}},
    '6': {'class_type': 'EmptySD3LatentImage', 'inputs': {'width': 512, 'height': 512, 'batch_size': 1}},
    '7': {'class_type': 'KSampler', 'inputs': {'seed': 42, 'steps': 4, 'cfg': 1.0, 'sampler_name': 'euler', 'scheduler': 'simple', 'denoise': 1.0, 'model': ['1', 0], 'positive': ['4', 0], 'negative': ['5', 0], 'latent_image': ['6', 0]}},
    '8': {'class_type': 'VAEDecode', 'inputs': {'samples': ['7', 0], 'vae': ['3', 0]}},
    '9': {'class_type': 'SaveImage', 'inputs': {'filename_prefix': 'flux1_workflow', 'images': ['8', 0]}},
}
save_workflow(flux1_workflow, 'flux1')

In [ ]:
# Flux.2 Klein workflow JSON only.
flux2_workflow = {
    '1': {'class_type': 'UNETLoader', 'inputs': {'unet_name': 'flux-2-klein-4b-fp8.safetensors', 'weight_dtype': 'default'}},
    '2': {'class_type': 'CLIPLoader', 'inputs': {'clip_name': 'qwen_3_4b.safetensors', 'type': 'flux2', 'device': 'default'}},
    '3': {'class_type': 'VAELoader', 'inputs': {'vae_name': 'flux2-vae.safetensors'}},
    '4': {'class_type': 'CLIPTextEncode', 'inputs': {'text': 'A ceramic cup of Vietnamese coffee on a wooden table, soft morning light', 'clip': ['2', 0]}},
    '5': {'class_type': 'CLIPTextEncode', 'inputs': {'text': '', 'clip': ['2', 0]}},
    '6': {'class_type': 'EmptyFlux2LatentImage', 'inputs': {'width': 1024, 'height': 768, 'batch_size': 1}},
    '7': {'class_type': 'RandomNoise', 'inputs': {'noise_seed': 42}},
    '8': {'class_type': 'CFGGuider', 'inputs': {'cfg': 1.0, 'model': ['1', 0], 'positive': ['4', 0], 'negative': ['5', 0]}},
    '9': {'class_type': 'KSamplerSelect', 'inputs': {'sampler_name': 'euler'}},
    '10': {'class_type': 'Flux2Scheduler', 'inputs': {'steps': 4, 'width': 1024, 'height': 768}},
    '11': {'class_type': 'SamplerCustomAdvanced', 'inputs': {'noise': ['7', 0], 'guider': ['8', 0], 'sampler': ['9', 0], 'sigmas': ['10', 0], 'latent_image': ['6', 0]}},
    '12': {'class_type': 'VAEDecode', 'inputs': {'samples': ['11', 0], 'vae': ['3', 0]}},
    '13': {'class_type': 'SaveImage', 'inputs': {'filename_prefix': 'flux2_workflow', 'images': ['12', 0]}},
}
save_workflow(flux2_workflow, 'flux2')

In [ ]:
# Qwen-Image workflow JSON only.
qwen_workflow = {
    '1': {'class_type': 'UNETLoader', 'inputs': {'unet_name': 'qwen_image_fp8_e4m3fn.safetensors', 'weight_dtype': 'default'}},
    '2': {'class_type': 'CLIPLoader', 'inputs': {'clip_name': 'qwen_2.5_vl_7b_fp8_scaled.safetensors', 'type': 'qwen_image', 'device': 'default'}},
    '3': {'class_type': 'VAELoader', 'inputs': {'vae_name': 'qwen_image_vae.safetensors'}},
    '4': {'class_type': 'CLIPTextEncode', 'inputs': {'text': 'A clean poster saying KAGGLE AI in blue letters, a robot holding a notebook', 'clip': ['2', 0]}},
    '5': {'class_type': 'CLIPTextEncode', 'inputs': {'text': 'blurry, distorted text', 'clip': ['2', 0]}},
    '6': {'class_type': 'EmptySD3LatentImage', 'inputs': {'width': 512, 'height': 512, 'batch_size': 1}},
    '7': {'class_type': 'KSampler', 'inputs': {'seed': 42, 'steps': 4, 'cfg': 4.0, 'sampler_name': 'euler', 'scheduler': 'simple', 'denoise': 1.0, 'model': ['1', 0], 'positive': ['4', 0], 'negative': ['5', 0], 'latent_image': ['6', 0]}},
    '8': {'class_type': 'VAEDecode', 'inputs': {'samples': ['7', 0], 'vae': ['3', 0]}},
    '9': {'class_type': 'SaveImage', 'inputs': {'filename_prefix': 'qwen_workflow', 'images': ['8', 0]}},
}
save_workflow(qwen_workflow, 'qwen_image')

## Assignment 2.9 — ComfyUI Flux Kontext workflow API JSON

Use the Kaggle Dataset tryon-images only as the two workflow inputs: person.jpg (image 1) and clothing.jpg (image 2). The API JSON describes the reference-image graph and prompt for changing the outfit while preserving the person's face, hair, pose and background.

No Flux model loading, T4 inference, or image decoding is needed here. The exported file is /kaggle/working/flux_kontext_tryon_api.json.

In [ ]:
# ComfyUI API workflow only: two reference images, no inference.
import json
from pathlib import Path
tryon_workflow = {
    '1': {'class_type': 'UNETLoader', 'inputs': {
        'unet_name': 'flux1-dev-kontext_fp8_scaled.safetensors',
        'weight_dtype': 'default'}},
    '2': {'class_type': 'DualCLIPLoader', 'inputs': {
        'clip_name1': 'clip_l.safetensors',
        'clip_name2': 't5xxl_fp8_e4m3fn_scaled.safetensors',
        'type': 'flux'}},
    '3': {'class_type': 'VAELoader', 'inputs': {'vae_name': 'ae.safetensors'}},
    '4': {'class_type': 'LoadImage', 'inputs': {'image': 'person.jpg', 'upload': 'image'}},
    '5': {'class_type': 'LoadImage', 'inputs': {'image': 'clothing.jpg', 'upload': 'image'}},
    '6': {'class_type': 'ImageStitch', 'inputs': {
        'image1': ['4', 0], 'image2': ['5', 0],
        'direction': 'right', 'match_image_size': True,
        'spacing': 0, 'color': 'white'}},
    '7': {'class_type': 'FluxKontextImageScale', 'inputs': {'image': ['6', 0]}},
    '8': {'class_type': 'VAEEncode', 'inputs': {'pixels': ['7', 0], 'vae': ['3', 0]}},
    '9': {'class_type': 'CLIPTextEncode', 'inputs': {
        'text': 'Make image 1 person wear the blue T-shirt from image 2. Keep face, hair, pose, hands and background.',
        'clip': ['2', 0]}},
    '10': {'class_type': 'FluxGuidance', 'inputs': {'conditioning': ['9', 0], 'guidance': 2.5}},
    '11': {'class_type': 'ReferenceLatent', 'inputs': {
        'conditioning': ['10', 0], 'latent': ['8', 0]}},
    '12': {'class_type': 'ConditioningZeroOut', 'inputs': {'conditioning': ['11', 0]}},
    '13': {'class_type': 'EmptySD3LatentImage', 'inputs': {
        'width': 1024, 'height': 512, 'batch_size': 1}},
    '14': {'class_type': 'KSampler', 'inputs': {
        'seed': 42, 'steps': 8, 'cfg': 1.0,
        'sampler_name': 'euler', 'scheduler': 'simple', 'denoise': 1.0,
        'model': ['1', 0], 'positive': ['11', 0], 'negative': ['12', 0],
        'latent_image': ['13', 0]}},
    '15': {'class_type': 'VAEDecode', 'inputs': {'samples': ['14', 0], 'vae': ['3', 0]}},
    '16': {'class_type': 'SaveImage', 'inputs': {
        'filename_prefix': 'flux_kontext_tryon', 'images': ['15', 0]}},
}
out = Path('/kaggle/working/flux_kontext_tryon_api.json')
out.write_text(json.dumps(tryon_workflow, indent=2))
print('WORKFLOW_JSON:', out)
print('Inputs: person.jpg + clothing.jpg; execution is intentionally not run.')

In [ ]:
# No package installation is needed for the ComfyUI JSON deliverable.

In [ ]:
# The workflow JSON was exported above; no inference is required.
print('COMFY_WORKFLOW_ONLY: flux_kontext_tryon_api.json')

In [ ]:
# No model cleanup is needed because this section does not load or run a model.